### Joining and rearranging nested datasets

Here is a quick example using sample data to demonstrate joining dataframes with nested fields into flattened dataframes, then aggregating them back into nested frames.

In [1]:
from pyspark.sql.functions import col, explode, collect_list, first, struct
import requests
import json

### Load up some example data

In [2]:
# Roughly approximates a maf file
gene = sqlContext.read.json("data/gene.json")
gene.printSchema()
gene.show()

root
 |-- case_submitter_id: long (nullable = true)
 |-- geneid: long (nullable = true)
 |-- name: string (nullable = true)

+-----------------+------+----+
|case_submitter_id|geneid|name|
+-----------------+------+----+
|                1|     1| AAA|
|                3|     2| BBB|
|                2|     2| BBB|
|                4|     3| CCC|
|                9|     4| DDD|
|                1|     4| DDD|
|                5|     2| BBB|
|                9|     3| CCC|
|                6|     4| DDD|
|                7|     4| DDD|
|                8|     2| BBB|
+-----------------+------+----+



In [3]:
# Roughly approximates a case doc
case = sqlContext.read.json("data/case.json")
case.printSchema()
case.show()

root
 |-- demographic: struct (nullable = true)
 |    |-- gender: string (nullable = true)
 |-- diagnoses: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- submitter_id: long (nullable = true)

+-----------+---------+------------+
|demographic|diagnoses|submitter_id|
+-----------+---------+------------+
|     [male]|      [0]|           1|
|   [female]|      [0]|           2|
|     [male]|      [0]|           3|
|     [male]|      [0]|           4|
|   [female]|      [0]|           5|
|     [male]|      [0]|           6|
|   [female]|      [0]|           7|
|     [male]|      [0]|           8|
|   [female]|      [0]|           9|
|   [female]|      [0]|          10|
+-----------+---------+------------+



In [4]:
ssm = sqlContext.read.json("data/ssm.json")
ssm.printSchema()
ssm.show()

root
 |-- cases: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- conseqeunce: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- consqid: long (nullable = true)
 |-- mutid: long (nullable = true)
 |-- pos: long (nullable = true)
 |-- type: string (nullable = true)

+---------------+-----------+-----+---+----+
|          cases|conseqeunce|mutid|pos|type|
+---------------+-----------+-----+---+----+
|[1, 8, 9, 5, 3]| [[1], [2]]|    1|123| ins|
|            [2]| [[3], [4]]|    1|521| del|
|      [3, 8, 1]| [[2], [6]]|    1|933|repl|
|      [9, 8, 3]| [[5], [7]]|    1|323| ins|
|         [3, 5]| [[8], [3]]|    1|423| ins|
+---------------+-----------+-----+---+----+



### Now, let's join gene and case docs

In [5]:
case_gene = gene.join(case, gene.case_submitter_id == case.submitter_id, 'left')
case_gene.printSchema()
case_gene.show()

root
 |-- case_submitter_id: long (nullable = true)
 |-- geneid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- demographic: struct (nullable = true)
 |    |-- gender: string (nullable = true)
 |-- diagnoses: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- submitter_id: long (nullable = true)

+-----------------+------+----+-----------+---------+------------+
|case_submitter_id|geneid|name|demographic|diagnoses|submitter_id|
+-----------------+------+----+-----------+---------+------------+
|                1|     1| AAA|     [male]|      [0]|           1|
|                3|     2| BBB|     [male]|      [0]|           3|
|                2|     2| BBB|   [female]|      [0]|           2|
|                4|     3| CCC|     [male]|      [0]|           4|
|                9|     4| DDD|   [female]|      [0]|           9|
|                1|     4| DDD|     [male]|      [0]|           1|
|                5|     2| BBB|   [female]|      [0]|     

### Need to reorganize this flat mapping to the case struct

In [6]:
reorg = case_gene.select(struct(*case.columns).alias('case'), *gene.columns)
reorg.printSchema()
reorg.show()

root
 |-- case: struct (nullable = false)
 |    |-- demographic: struct (nullable = true)
 |    |    |-- gender: string (nullable = true)
 |    |-- diagnoses: array (nullable = true)
 |    |    |-- element: long (containsNull = true)
 |    |-- submitter_id: long (nullable = true)
 |-- case_submitter_id: long (nullable = true)
 |-- geneid: long (nullable = true)
 |-- name: string (nullable = true)

+--------------------+-----------------+------+----+
|                case|case_submitter_id|geneid|name|
+--------------------+-----------------+------+----+
|[[male],WrappedAr...|                1|     1| AAA|
|[[male],WrappedAr...|                3|     2| BBB|
|[[female],Wrapped...|                2|     2| BBB|
|[[male],WrappedAr...|                4|     3| CCC|
|[[female],Wrapped...|                9|     4| DDD|
|[[male],WrappedAr...|                1|     4| DDD|
|[[female],Wrapped...|                5|     2| BBB|
|[[female],Wrapped...|                9|     3| CCC|
|[[male],Wrapped

### Then groupBy the genes and aggregate the cases with collect_list to turn them into arrays of docs

In [7]:
gene_centric = reorg.groupBy('geneid','name').agg(collect_list('case').alias('case'))
gene_centric.printSchema()
gene_centric.show()

root
 |-- geneid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- case: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- demographic: struct (nullable = true)
 |    |    |    |-- gender: string (nullable = true)
 |    |    |-- diagnoses: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- submitter_id: long (nullable = true)

+------+----+--------------------+
|geneid|name|                case|
+------+----+--------------------+
|     1| AAA|[[[male],WrappedA...|
|     3| CCC|[[[male],WrappedA...|
|     2| BBB|[[[male],WrappedA...|
|     4| DDD|[[[female],Wrappe...|
+------+----+--------------------+



### Now do it again, but all at once (so the query planner can optimize further)

In [8]:
gene_centric = gene.join(case, gene.case_submitter_id == case.submitter_id, 'left')\
                    .select(struct(*case.columns).alias('case'), *gene.columns)\
                    .groupBy('geneid','name').agg(collect_list('case').alias('case'))
gene_centric.printSchema()
gene_centric.show()

root
 |-- geneid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- case: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- demographic: struct (nullable = true)
 |    |    |    |-- gender: string (nullable = true)
 |    |    |-- diagnoses: array (nullable = true)
 |    |    |    |-- element: long (containsNull = true)
 |    |    |-- submitter_id: long (nullable = true)

+------+----+--------------------+
|geneid|name|                case|
+------+----+--------------------+
|     1| AAA|[[[male],WrappedA...|
|     3| CCC|[[[male],WrappedA...|
|     2| BBB|[[[male],WrappedA...|
|     4| DDD|[[[female],Wrappe...|
+------+----+--------------------+



### We now have a df with the desired schema ready to export to es!!!
Let's export it now

In [9]:
# Start from a fresh index
requests.delete('http://localhost:9200/example').json()

{u'acknowledged': True}

In [10]:
gene_centric.write.format('org.elasticsearch.spark.sql')\
            .option('es.mapping.id','geneid')\
            .save('example/gene')

In [11]:
requests.get('http://localhost:9200/example/_count').json()

{u'_shards': {u'failed': 0, u'successful': 5, u'total': 5}, u'count': 4}

In [12]:
requests.get('http://localhost:9200/example/_search').json()['hits']['hits'][0]['_source']

{u'case': [{u'demographic': {u'gender': u'female'},
   u'diagnoses': [0],
   u'submitter_id': 9},
  {u'demographic': {u'gender': u'male'},
   u'diagnoses': [0],
   u'submitter_id': 1},
  {u'demographic': {u'gender': u'male'},
   u'diagnoses': [0],
   u'submitter_id': 6},
  {u'demographic': {u'gender': u'female'},
   u'diagnoses': [0],
   u'submitter_id': 7}],
 u'geneid': 4,
 u'name': u'DDD'}

# Exploding lists and aggregating with collect_list

This method doesn't seem as good as the one above

With dataframes produced from json objects, there are typically many fields that are arrays. It is not possible to join on arrays directly, but we may pivot the array using `explode` creating denormalized columns that
are able to be joined on. 
After joining, it may be desired to revert to the previous structure.
To do this, a `groupBy` operation followed by a `collect_list` aggregation may be applied.

In [18]:
# The exploded case df has the array value of genes pivoted
exploded = case.select(col('submitter_id'),explode(col('geneid')).alias('gene'))
# We can 'undo' this action with groupby and collect_list agg
imploded = exploded.groupBy(col('submitter_id')).agg(collect_list('geneid'))
imploded.orderBy('submitter_id').show(), case.show()

AnalysisException: u"cannot resolve '`geneid`' given input columns: [demographic, diagnoses, submitter_id];"